# HW2P2: Image Recognition and Verification

> This is the second homework assignment for **11-785: Introduction to Deep Learning**, which focuses on the task of **image verification**. In this assignment, we first train a custom **convolutional neural network (CNN)** to perform **face classification** over **8,631 distinct identities**.
>
> Once the model is trained, we use it to extract **face embeddings** for images. These embeddings are then compared across pairs of images to determine whether the two faces belong to the **same identity** or **different identities**, thereby solving the image verification problem.


## 📅 Submission Schedule
- ✅ **Checkpoint:** October 2, 2026 @ 11:59 PM (EST)  
- 🏁 **Final Submission:** October 9, 2026 @ 11:59 PM (EST)  
  *Slack deadline: October 23, 2026 @ 11:59 PM (EST)*  
- 💻 **Code Submission:**
  - Due: October 11, 2026 @ 11:59PM (EST)
  - Closes: October 23, 2026 @ 11:59 PM (EST) *or day-of via Slack*

## Requirement Acknowledgement

Setting the flag below to **True** indicates that you have read, understood, and agree to comply with **all** of the following requirements:

1. **Slack Days Usage**  
   Slack days may be used **only** for the **P2 Final submission**, *not* for the checkpoint.  
   Specifically, slack days may be applied to submit **final P2 Kaggle scores** to the **Slack Kaggle Competition**, at the expense of your available slack days.

2. **Final Code Submission Deadline**  
   The final **Gradescope code submission** is due **within 48 hours after** the Kaggle competition deadline.

3. **Kaggle Username & Score Verification**  
   You must provide your **Kaggle username** below. We will use it to retrieve your official **PRIVATE leaderboard** score.  
   Minor score or output variance is acceptable; however, any discrepancies caused by **modifications to the required submission code**
   (including the notebook’s final submission cell) will result in an **Academic Integrity Violation (AIV)**.

4. **Model Implementation Restrictions**  
   You are **not permitted** to use pre-trained or pre-loaded models (e.g., from Hugging Face or similar libraries).  
   You **may** implement models described in research papers or articles, but they **must** be implemented **from scratch**
   using fundamental PyTorch components (e.g., `Linear`, `Conv2d`, etc.).

5. **Data Usage Restrictions**  
   The use of **any external data or datasets** is strictly prohibited at **any stage** of this assignment.

6. **Collaboration Policy**  
   You may collaborate with teammates to run experiments or ablations.  
   However, you must submit **your own code** and **your own results**.

7. **Academic Integrity**  
   Failure to comply with any of the above requirements will be treated as an **Academic Integrity Violation (AIV)**.

8. **Late Submissions**  
   Late submissions **must** be submitted through the **Slack Kaggle Competition** (see write-up for details).  
   Submissions made to the **regular Kaggle competition** after the original deadline will **not** be considered,
   regardless of remaining slack days.
   
   Gradescope code submission will close at **the same time** as the Slack Kaggle Competition, rather than remaining open for an additional 48 hours.


In [1]:
# @title Click to acknowledge then run cell!
ACKNOWLEDGED = True # @param {"type":"boolean","placeholder":"False"}


## Welcome to the World of Convolutions!

In the previous homework, you explored **Multi-Layer Perceptrons (MLPs)**—the foundational building blocks of deep learning. You saw how MLPs can learn patterns from data, predict phonemes from speech, and even approximate complex functions. However, despite their expressive power, MLPs are not well-suited for image data. Flattening an image into a one-dimensional vector discards crucial **spatial information**, such as how neighboring pixels relate to one another.

This limitation motivates the use of **Convolutional Neural Networks (CNNs)**. CNNs preserve spatial structure by operating directly on image grids, allowing models to learn meaningful visual patterns. In doing so, they bring us closer to designing systems that can truly *see* and interpret the visual world.

In this homework, you will dive into CNNs through **face classification** and **face verification** tasks. In face classification, the goal is to identify a face as belonging to one of several known identities. In contrast, face verification asks a different question: *Do these two images belong to the same person?*—even if that person was never seen during training.

To make this concrete, imagine being given a photo of a friend and a group photo. A classifier would attempt to name your friend, while a verification model would simply determine whether the two images depict the same individual. CNNs enable this by learning **hierarchical features**: early layers capture edges and textures, while deeper layers encode higher-level concepts such as facial structures and distinctive features.

A key theme of this assignment is understanding how **loss functions** shape model behavior. Classification models typically rely on **Cross-Entropy Loss**, which encourages correct label prediction. Verification tasks, however, depend on learning meaningful **embeddings**, where similarity in feature space reflects identity similarity. Loss functions such as **Triplet Loss** or **ArcFace Loss** are designed for this purpose, pulling embeddings of the same person closer together while pushing different identities apart. A well-trained verification model can even distinguish between visually similar individuals—such as siblings—while remaining robust to changes in pose, lighting, or expression.

Beyond theory, this homework emphasizes **practical deep learning skills**. You will preprocess image data, apply data augmentation techniques (e.g., random cropping or flipping), and experiment with modern CNN architectures such as **ResNet** or **ConvNeXt**. Along the way, you will discover that a model optimized for classification is not automatically suitable for verification—highlighting the importance of aligning architectural choices and loss functions with the task at hand.

By the end of this assignment, you will have built a robust face recognition system inspired by real-world applications, from smartphone unlocking to identity verification systems. Let’s get started.

# **SET-UP**


Click [**HERE**](https://www.kaggle.com/t/d025d75eb26d436da206a8e7aaa38235) to join the competition first!

## Portable setup: Kaggle, local, PSC, and Colab


### Runtime settings
Run this notebook from top to bottom. Change machine settings here; keep experiment settings in the Config cell. See `PORTABILITY.md` for setup and checkpoint transfer instructions. Missing data is downloaded automatically on local, PSC, and Colab. Package installation is opt-in; PSC uses the class environment.


In [2]:
import os
import sys
import importlib.util
import subprocess
from pathlib import Path

PLATFORM = os.environ.get("HW2_PLATFORM", "auto")  # auto, kaggle, local, psc, colab
if PLATFORM == "auto":
    if Path("/kaggle/input").exists():
        PLATFORM = "kaggle"
    elif os.environ.get("SLURM_JOB_ID") or Path("/jet/home").exists():
        PLATFORM = "psc"
    elif importlib.util.find_spec("google") is not None and importlib.util.find_spec("google.colab") is not None:
        PLATFORM = "colab"
    else:
        PLATFORM = "local"
if PLATFORM not in {"kaggle", "local", "psc", "colab"}:
    raise ValueError(f"Unknown platform: {PLATFORM}")
if PLATFORM == "psc" and not os.environ.get("LOCAL"):
    raise RuntimeError("Start a PSC compute-node session first: $LOCAL must be set.")

# Local and PSC: load .env from the kernel's working directory.
# Set HW2_ENV_FILE in the environment (or edit ENV_FILE) for another location.
# Existing environment variables take precedence; secrets are never logged.
ENV_FILE = Path(os.environ.get("HW2_ENV_FILE", ".env")).expanduser()
if PLATFORM in {"local", "psc"}:
    if "HW2_ENV_FILE" in os.environ and not ENV_FILE.is_file():
        raise FileNotFoundError(f"Configured .env file does not exist: {ENV_FILE}")
    if ENV_FILE.is_file():
        try:
            from dotenv import load_dotenv
        except ImportError as exc:
            raise ImportError(
                "Loading .env requires python-dotenv. Install it in your notebook kernel "
                "environment with %pip install python-dotenv. On PSC, use a personal "
                "environment or export the variables before launching Jupyter instead."
            ) from exc
        load_dotenv(dotenv_path=ENV_FILE, override=False, interpolate=False)

_data_defaults = {
    "kaggle": "/kaggle/input/competitions/hw-2-p-2-fall-2026-student-competition/hw2p2_data",
    "local": "./hw2p2_data",
    "psc": str(Path(os.environ.get("LOCAL", "/local")) / "hw2p2_data"),
    "colab": "/content/hw2p2_data",
}
_output_defaults = {
    "kaggle": "/kaggle/working/checkpoints",
    "local": "./checkpoints",
    "psc": str(Path.home() / "hw2p2_checkpoints"),
    "colab": "/content/checkpoints",
}
runtime = {
    "platform": PLATFORM,
    "data_root": os.environ.get("HW2_DATA_ROOT", _data_defaults[PLATFORM]),
    "checkpoint_root": os.environ.get("HW2_CHECKPOINT_ROOT", _output_defaults[PLATFORM]),
    "num_workers": int(os.environ.get("HW2_NUM_WORKERS", 1 if PLATFORM == "local" else 4)),
    "persistent_workers": False,
    "prefetch_factor": 1,
    "visualize": False,
    "install_packages": False,  # Enable once if dependencies are missing; never on PSC.
    "download_data": True,      # Download missing data on local/PSC/Colab; reuse existing data.
    "mount_drive": False,       # Colab only; then set mirror_root below to a Drive folder.
    "mirror_root": os.environ.get("HW2_MIRROR_ROOT") or None,
    "resume_from": os.environ.get("HW2_RESUME_FROM") or None,
    "wandb_mode": os.environ.get("WANDB_MODE", "online"),  # online, offline, disabled
    "wandb_project": "IdlF26Hw2P2",
    "wandb_entity": os.environ.get("WANDB_ENTITY") or None,
    "upload_checkpoints": False,  # Upload saved checkpoints via W&B when online.
}
if runtime["wandb_mode"] not in {"online", "offline", "disabled"}:
    raise ValueError("wandb_mode must be online, offline, or disabled")
if runtime["num_workers"] < 0:
    raise ValueError("num_workers must be nonnegative")
for key in ("data_root", "checkpoint_root", "resume_from", "mirror_root"):
    if runtime[key]:
        runtime[key] = str(Path(runtime[key]).expanduser().resolve())
if runtime["mount_drive"]:
    if PLATFORM != "colab":
        raise ValueError("mount_drive is only supported on Colab")
    from google.colab import drive
    drive.mount("/content/drive")
if runtime["install_packages"]:
    if PLATFORM == "psc":
        raise RuntimeError("Use the provided PSC environment; do not install into it.")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "wandb==0.23.1",
        "pytorch_metric_learning", "torchsummary", "torchinfo", "kaggle==1.8.0",
        "python-dotenv", "numpy", "pandas", "scipy", "scikit-learn", "matplotlib", "pillow", "tqdm"])
print("Platform:", PLATFORM)
print("Dataset:", runtime["data_root"])
print("Checkpoint root:", runtime["checkpoint_root"])
print("DataLoader workers:", runtime["num_workers"])


Platform: local
Dataset: /home/lukas/CMU/IDL/HW2/HW2P2/hw2p2_data
Checkpoint root: /home/lukas/CMU/IDL/HW2/HW2P2/checkpoints
DataLoader workers: 1


### Dataset and storage
- **Kaggle:** Attach the competition dataset through Add Input. Override `data_root` if its mount path differs.
- **Local:** Install CUDA-compatible PyTorch/torchvision in your notebook environment. Data is downloaded automatically if missing; use an SSD for storage.
- **PSC:** Activate the supplied class environment and run Jupyter on an allocated GPU node. Data defaults to `$LOCAL/hw2p2_data`; checkpoints default to your persistent home directory. Missing data is downloaded automatically on each new node.
- **Colab:** Select a GPU runtime. Download data to `/content`. For durable checkpoints, enable `mount_drive` and set `mirror_root` to a folder under `/content/drive/MyDrive`, or enable W&B checkpoint uploads.

`download_data=True` is the default: check for the expected dataset layout, then download and extract it only if missing. Existing data is reused without Kaggle authentication. On Kaggle, attach the competition dataset through Add Input. Set `download_data=False` to require manually staged data. Checkpoints are saved after each completed epoch; an interrupted epoch is repeated on resume.


### Credentials and automatic download
Use environment variables `KAGGLE_API_TOKEN` / `KAGGLE_USERNAME` and `WANDB_API_KEY`, or platform secrets. Kaggle secret names remain `KaggleApiToken` and `WANDB_API_TOKEN`; Colab accepts those names too. Credentials are only needed for enabled services. Never put tokens in the notebook.

On local and PSC runs, `.env` in the kernel working directory is loaded automatically before runtime settings. Set `HW2_ENV_FILE` to use another path. Existing environment variables win. Use `KAGGLE_API_TOKEN`, `KAGGLE_USERNAME`, and `WANDB_API_KEY` as the names in `.env`. Restart the kernel after changing credentials so previously loaded values do not take precedence.


In [3]:
def get_secret(env_name, platform_name=None):
    value = os.environ.get(env_name)
    if value:
        return value
    if PLATFORM == "kaggle":
        from kaggle_secrets import UserSecretsClient
        try:
            return UserSecretsClient().get_secret(platform_name or env_name)
        except Exception:
            return None
    if PLATFORM == "colab":
        from google.colab import userdata
        try:
            return userdata.get(platform_name or env_name)
        except Exception:
            return None
    return None

def get_kaggle_api():
    token = get_secret("KAGGLE_API_TOKEN", "KaggleApiToken")
    if token:
        os.environ["KAGGLE_API_TOKEN"] = token
    import kaggle
    return kaggle.api

# Lazy authentication: training from existing data does not require Kaggle credentials.
api = None
_data_root = Path(runtime["data_root"])
_required = ["cls_data/train/images", "cls_data/train/labels.txt", "cls_data/dev/images",
             "cls_data/dev/labels.txt", "cls_data/test/images", "ver_data", "val_pairs.txt", "test_pairs.txt"]
if runtime["download_data"] and not all((_data_root / name).exists() for name in _required):
    if PLATFORM == "kaggle":
        raise ValueError("Attach the competition dataset using Add Input on Kaggle.")
    if _data_root.name != "hw2p2_data":
        raise ValueError("For automatic extraction, data_root must end with hw2p2_data.")
    import zipfile
    _data_root.parent.mkdir(parents=True, exist_ok=True)
    api = get_kaggle_api()
    competition = "hw-2-p-2-fall-2026-student-competition"
    api.competition_download_files(competition, path=str(_data_root.parent))
    archive = _data_root.parent / f"{competition}.zip"
    with zipfile.ZipFile(archive) as bundle:
        bundle.extractall(_data_root.parent)
    archive.unlink()
missing = [name for name in _required if not (_data_root / name).exists()]
if missing:
    raise FileNotFoundError(f"Dataset incomplete at {_data_root}: {missing}. Set data_root or enable download_data.")


# Imports

In [4]:
import torch
from torch.utils.data import Dataset
from torch.utils.data import DataLoader
import torch.nn.functional as F
import torch.nn as nn
import torch.optim as optim
from torchsummary import summary
import torchvision
from torchvision.io import decode_image
from torchvision.utils import make_grid
from torchvision import transforms
import torchvision.transforms.v2 as T
import os
import gc
from tqdm import tqdm
from tqdm.auto import tqdm
from PIL import Image
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score
from sklearn import metrics as mt
from scipy.optimize import brentq
from scipy.interpolate import interp1d
import glob
import wandb
import matplotlib.pyplot as plt
from pytorch_metric_learning import samplers
import csv
import pandas as pd

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print("Device: ", DEVICE)


/home/lukas/miniconda3/envs/idlf26/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device:  cuda


# Config

Experiment settings below stay the same when moving a run. Machine paths and loader settings live in `runtime`. Keep the original total `epochs` when resuming so the scheduler continues on the same schedule.


In [ ]:
config = {
    'seed': 42,
    'train_image_fraction': 0.2, # (0, 1]: fraction per training class; 0.2 = 10 of 50 images
    'display_name': 'resnet18-2342-c64-512-prelu full',
    'description': 'compare 20% performance to full performance',
    'batch_size': 64, # Increase this if your GPU can handle it
    'lr': 1e-3,
    'epochs': 20, # 20 epochs is recommended ONLY for the early submission - you will have to train for much longer typically.
    'num_classes': 1726, # full: 8631, # 20%: 1726 Dataset contains 8631 classes for classification, reduce this number if you want to train on a subset, but only for train dataset and not on val dataset
    'augument': True,
    # Include other parameters as needed.
    'label_smoothing': 0.1,
    'weight_decay': 5e-4,
}

import random
random.seed(config["seed"])
np.random.seed(config["seed"])
torch.manual_seed(config["seed"])
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(config["seed"])

def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % 2**32
    random.seed(worker_seed)
    np.random.seed(worker_seed)

loader_generator = torch.Generator().manual_seed(config["seed"])
loader_options = {
    "num_workers": runtime["num_workers"],
    "pin_memory": DEVICE == "cuda",
    "worker_init_fn": seed_worker,
    "generator": loader_generator,
}
if runtime["num_workers"] > 0:
    loader_options.update(persistent_workers=runtime["persistent_workers"],
                          prefetch_factor=runtime["prefetch_factor"])
print("PyTorch:", torch.__version__, "torchvision:", torchvision.__version__)
if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name())
else:
    print("CPU selected: use a CUDA-enabled environment for training.")


PyTorch: 2.13.0+cu130 torchvision: 0.28.0+cu130
GPU: NVIDIA RTX A5500 Laptop GPU


# Data Augumentations

In [6]:
def create_transforms(image_size: int = 112, augment: bool = True) -> T.Compose:
    """Create transform pipeline for face recognition."""

    # Step 1: Basic transformations
    transform_list = [
        # Resize the image to the desired size (image_size x image_size)
        T.Resize((image_size, image_size)),

        # Convert PIL Image to tensor
        T.ToTensor(),

        # Convert image to float32 and scale the pixel values to [0, 1]
        T.ToDtype(torch.float32, scale=True),
    ]

    # Step 2: Data augmentation (optional, based on `augment` argument)
    if augment:  # This block will be executed if `augment=True`
        # TODO: Add transformations for data augmentation (e.g., random horizontal flip, rotation, etc.)
        # HINT: What transforms help faces look more varied?
        # Think: Does a horizontally flipped face still look like the same person?
        # What about small rotations or color changes?
        # Example:
        transform_list.extend([
            T.RandomHorizontalFlip(p=0.5),
            
            T.RandomApply([
                T.RandomAffine(
                    degrees=10,                 # rotation between -10° and +10°
                    translate=(0.05, 0.05),     # shift by up to 5%
                    scale=(0.95, 1.05),
                    interpolation=T.InterpolationMode.BILINEAR,
                    fill=0.5,                  # gray for [0, 1] images
                ),
            ], p=0.5),

            T.ColorJitter(
                brightness=0.2,
                contrast=0.2,
                saturation=0.1,
                hue=0.05
            ),
        ])

    # Step 3: Standard normalization for image recognition tasks
    # The Normalize transformation requires mean and std values for each channel (R, G, B).
    # Here, we are normalizing the pixel values to have a mean of 0.5 and std of 0.5 for each channel.
    transform_list.extend([
        T.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])  # Standard mean and std for face recognition tasks
    ])

    # Return the composed transformation pipeline
    return T.Compose(transform_list)


In [7]:
train_transforms = create_transforms(augment=config['augument'])
val_transforms   = create_transforms(augment=False)

/home/lukas/miniconda3/envs/idlf26/lib/python3.13/site-packages/torchvision/transforms/v2/_deprecated.py:42: UserWarning: The transform `ToTensor()` is deprecated and will be removed in a future release. Instead, please use `v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)])`.Output is equivalent up to float precision.
  warnings.warn(


# Datasets and DataLoaders

## Classification Datasets and Dataloaders

In [8]:
class ImageDataset(Dataset):
    """Custom dataset for loading images with optional labels."""

    def __init__(self, root, transform, num_classes=None, preload=False, image_fraction=1.0, seed=42):
        """
        Args:
            root (str): Dataset root directory.
                Expected structure:
                  - labeled:
                        root/
                          images/
                          labels.txt   (img label)
                  - unlabeled:
                        root/
                          images/
            transform (callable): Transform applied to images.
            num_classes (int, optional): Number of classes to keep (labeled only).
            preload (bool): Preload images into memory.
            image_fraction (float): Fraction per class in (0, 1], rounded down, minimum one.
            seed (int): Seed for reproducible image selection.
        """
        if not 0 < image_fraction <= 1:
            raise ValueError("image_fraction must be in (0, 1]")

        self.root = root
        self.transform = transform
        self.preload = preload

        self.image_paths = []
        self.labels = None      # None → unlabeled dataset
        self.classes = None
        self.images = []

        labels_file = os.path.join(self.root, "labels.txt")
        images_dir = os.path.join(self.root, "images")

        # --------------------------------------------------
        # Detect labeled vs unlabeled dataset
        # --------------------------------------------------
        has_labels = os.path.exists(labels_file)

        if has_labels:
            self.labels = []
            self.classes = set()

            with open(labels_file, "r") as f:
                lines = f.readlines()

            # Sort by label (keeps your original behavior)
            lines = sorted(lines, key=lambda x: int(x.strip().split(" ")[-1]))

            all_labels = sorted(set(int(line.strip().split(" ")[1]) for line in lines))

            if num_classes is not None:
                selected_classes = set(all_labels[:num_classes])
            else:
                selected_classes = set(all_labels)

            for line in tqdm(lines, desc="Loading labeled dataset"):
                img_path, label = line.strip().split(" ")
                label = int(label)

                if label in selected_classes:
                    self.image_paths.append(os.path.join(images_dir, img_path))
                    self.labels.append(label)
                    self.classes.add(label)

            # Select once before preloading; retain the same subset across epochs.
            if image_fraction < 1.0:
                indices_by_class = {}
                for index, label in enumerate(self.labels):
                    indices_by_class.setdefault(label, []).append(index)
                rng = random.Random(seed)
                selected_indices = []
                for indices in indices_by_class.values():
                    count = max(1, int(len(indices) * image_fraction))
                    selected_indices.extend(rng.sample(indices, count))
                selected_indices.sort()
                self.image_paths = [self.image_paths[i] for i in selected_indices]
                self.labels = [self.labels[i] for i in selected_indices]

            self.classes = sorted(self.classes)

            assert len(self.image_paths) == len(self.labels), "Images and labels mismatch!"

        else:
            # --------------------------------------------------
            # Unlabeled dataset: load all images from images/
            # --------------------------------------------------
            image_files = sorted(os.listdir(images_dir))

            for img in tqdm(image_files, desc="Loading unlabeled dataset"):
                self.image_paths.append(os.path.join(images_dir, img))

        # --------------------------------------------------
        # Preload images if requested
        # --------------------------------------------------
        if self.preload:
            self.images = [
                decode_image(p, mode="RGB") / 255.0
                for p in tqdm(self.image_paths, desc="Preloading images")
            ]

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        if self.preload:
            image = self.images[idx]
        else:
            image = decode_image(self.image_paths[idx], mode="RGB") / 255.0

        image = self.transform(image)

        # Return with or without label
        if self.labels is not None:
            return image, self.labels[idx]
        else:
            return image


In [9]:
gc.collect()
cls_data_dir = runtime['data_root'] + '/cls_data'

# Datasets
# TODO: Implement the datasets for training, validation, and testing
cls_train_dataset = ImageDataset(
    cls_data_dir + '/train', train_transforms,
    num_classes=config['num_classes'],
    image_fraction=config['train_image_fraction'], seed=config['seed'],
)
cls_val_dataset   = ImageDataset(cls_data_dir + '/dev', val_transforms, num_classes=config['num_classes'])
cls_test_dataset  = ImageDataset(cls_data_dir + '/test', val_transforms)

assert cls_train_dataset.classes == cls_val_dataset.classes, "Class mismatch!"

# Dataloaders
# TODO: Implement the dataloaders for training, validation, and testing
cls_train_loader = DataLoader(
    cls_train_dataset,
    batch_size=config["batch_size"],
    shuffle=True,
    **loader_options,
)
cls_val_loader = DataLoader(
    cls_val_dataset,
    batch_size=config["batch_size"],
    shuffle=False,
    **loader_options,
)
cls_test_loader = DataLoader(
    cls_test_dataset,
    batch_size=config["batch_size"],
    shuffle=False,
    **loader_options,
)


Loading unlabeled dataset: 100%|██████████| 43097/43097 [00:00<00:00, 2365250.70it/s]


## Verification Dataset and Datatloaders

In [10]:
class ImagePairDataset(Dataset):
    """Custom dataset for loading and transforming image pairs (labeled or unlabeled)."""

    def __init__(self, root, pairs_file, transform, preload=False):
        """
        Args:
            root (str): Path to the directory containing the images.
            pairs_file (str): File with:
                - img1 img2 label   (labeled)
                - img1 img2         (unlabeled)
            transform (callable): Transform to apply to images.
            preload (bool): Whether to preload images into memory.
        """
        self.root = root
        self.transform = transform
        self.preload = preload

        self.image1_paths = []
        self.image2_paths = []
        self.matches = None  # None → unlabeled dataset

        # Read file
        with open(pairs_file, "r") as f:
            lines = f.readlines()

        # Detect whether labels exist
        first_cols = lines[0].strip().split()
        has_labels = len(first_cols) == 3

        if has_labels:
            self.matches = []

        for line in tqdm(lines, desc="Loading dataset"):
            parts = line.strip().split()

            if has_labels:
                img1, img2, match = parts
                self.matches.append(int(match))
            else:
                img1, img2 = parts

            self.image1_paths.append(os.path.join(self.root, img1))
            self.image2_paths.append(os.path.join(self.root, img2))

        assert len(self.image1_paths) == len(self.image2_paths)
        if has_labels:
            assert len(self.matches) == len(self.image1_paths)

        # Preload images if requested
        if self.preload:
            self.image1_cache = [
                decode_image(p, mode="RGB") / 255.0
                for p in tqdm(self.image1_paths, desc="Preloading image1")
            ]
            self.image2_cache = [
                decode_image(p, mode="RGB") / 255.0
                for p in tqdm(self.image2_paths, desc="Preloading image2")
            ]

    def __len__(self):
        return len(self.image1_paths)

    def __getitem__(self, idx):
        # Load images
        if self.preload:
            img1 = self.image1_cache[idx]
            img2 = self.image2_cache[idx]
        else:
            img1 = decode_image(self.image1_paths[idx], mode="RGB") / 255.0
            img2 = decode_image(self.image2_paths[idx], mode="RGB") / 255.0

        img1 = self.transform(img1)
        img2 = self.transform(img2)

        # Return with or without label
        if self.matches is not None:
            return img1, img2, self.matches[idx]
        else:
            return img1, img2


In [11]:
gc.collect()
ver_data_dir = runtime['data_root'] + '/ver_data'

# Datasets
# TODO: Implement the datasets for validation and testing
ver_val_dataset  = ImagePairDataset(ver_data_dir, runtime['data_root'] + '/val_pairs.txt', val_transforms)
ver_test_dataset = ImagePairDataset(ver_data_dir, runtime['data_root'] + '/test_pairs.txt', val_transforms)

# Dataloader
# TODO: Implement the dataloaders for validation and testing
ver_val_loader   = DataLoader(
    ver_val_dataset,
    batch_size=config["batch_size"],
    shuffle=False,
    **loader_options,
)
ver_test_loader  = DataLoader(
    ver_test_dataset,
    batch_size=config["batch_size"],
    shuffle=False,
    **loader_options,
)


Loading dataset: 100%|██████████| 5000/5000 [00:00<00:00, 1645728.64it/s]


# EDA and Viz

In [12]:
# Double-check your dataset/dataloaders work as expected

print("Number of classes    : ", len(cls_train_dataset.classes))
print("No. of train images  : ", cls_train_dataset.__len__())
print("Shape of image       : ", cls_train_dataset[0][0].shape)
print("Batch size           : ", config['batch_size'])
print("Train batches        : ", cls_train_loader.__len__())
print("Val batches          : ", cls_val_loader.__len__())

# Feel free to print more things if needed

Number of classes    :  1726
No. of train images  :  86300
Shape of image       :  torch.Size([3, 112, 112])
Batch size           :  64
Train batches        :  1349
Val batches          :  135


### Classification Dataset Viz

In [13]:
def show_cls_dataset_samples(train_loader, val_loader, test_loader, samples_per_set=8, figsize=(10, 6)):
    """
    Display samples from train, validation, and test datasets side by side

    Args:
        train_loader: Training data loader
        val_loader: Validation data loader
        test_loader: Test data loader
        samples_per_set: Number of samples to show from each dataset
        figsize: Figure size (width, height)
    """
    def denormalize(x):
        """Denormalize images from [-1, 1] to [0, 1]"""
        return x * 0.5 + 0.5

    def get_samples(loader, n, labeled=True):
        """Get n samples from a dataloader"""
        batch = next(iter(loader))

        if labeled:
            return batch[0][:n], batch[1][:n]
        else:
            return batch[:n], None

    # Get samples from each dataset
    train_imgs, train_labels = get_samples(train_loader, samples_per_set, labeled=True)
    val_imgs, val_labels = get_samples(val_loader, samples_per_set, labeled=True)
    test_imgs, test_labels = get_samples(test_loader, samples_per_set, labeled=False)

    # Create figure
    fig, axes = plt.subplots(3, 1, figsize=figsize)

    # Plot each dataset
    for idx, (imgs, labels, title) in enumerate([
        (train_imgs, train_labels, 'Training Samples'),
        (val_imgs, val_labels, 'Validation Samples'),
        (test_imgs, test_labels, 'Test Samples')
    ]):

        # Create grid of images
        grid = make_grid(denormalize(imgs), nrow=8, padding=2)

        # Display grid
        axes[idx].imshow(grid.permute(1, 2, 0).cpu())
        axes[idx].axis('off')
        axes[idx].set_title(title, fontsize=10)

        # Add class labels below images (with smaller font)
        grid_width = grid.shape[2]
        imgs_per_row = min(8, samples_per_set)
        img_width = grid_width // imgs_per_row

        if labels is not None:
            for i, label in enumerate(labels):
                col = i % imgs_per_row

                if label < len(train_loader.dataset.classes):
                    class_name = train_loader.dataset.classes[label]
                else:
                    class_name = f"Class {label} (Unknown)"

                axes[idx].text(
                    col * img_width + img_width/2,
                    grid.shape[1] + 5,
                    class_name,
                    ha='center',
                    va='top',
                    fontsize=6,
                    rotation=45
                )

    plt.tight_layout()
    plt.show()

if runtime["visualize"]:
    show_cls_dataset_samples(cls_train_loader, cls_val_loader, cls_test_loader)


### Ver Dataset Viz

In [14]:
import matplotlib.pyplot as plt
import torch
from torchvision.utils import make_grid

def show_ver_dataset_samples(val_loader, samples_per_set=4, figsize=(12, 8)):
    """
    Display verification pairs from the validation dataset

    Args:
        val_loader: Validation data loader
        samples_per_set: Number of pairs to show from the dataset
        figsize: Figure size (width, height)
    """
    def denormalize(x):
        """Denormalize images from [-1, 1] to [0, 1]"""
        return x * 0.5 + 0.5

    def get_samples(loader, n):
        """Get n samples from a dataloader"""
        batch = next(iter(loader))
        return batch[0][:n], batch[1][:n], batch[2][:n]

    # Get samples from the validation dataset
    val_imgs1, val_imgs2, val_labels = get_samples(val_loader, samples_per_set)

    # Create figure and axis
    fig, ax = plt.subplots(1, 1, figsize=figsize)

    # Create grids for both images in each pair
    grid1 = make_grid(denormalize(val_imgs1), nrow=samples_per_set, padding=2)
    grid2 = make_grid(denormalize(val_imgs2), nrow=samples_per_set, padding=2)

    # Combine the grids vertically
    combined_grid = torch.cat([grid1, grid2], dim=1)

    # Display the combined grid
    ax.imshow(combined_grid.permute(1, 2, 0).cpu())
    ax.axis('off')
    ax.set_title('Validation Pairs', fontsize=10)

    # Determine dimensions for placing the labels
    grid_width = grid1.shape[2]
    img_width = grid_width // samples_per_set

    # Add match/non-match labels for each pair
    for i, label in enumerate(val_labels):
        match_text = "✓ Match" if label == 1 else "✗ Non-match"
        color = 'green' if label == 1 else 'red'

        # Define a background box for the label
        bbox_props = dict(
            boxstyle="round,pad=0.3",
            fc="white",
            ec=color,
            alpha=0.8
        )

        ax.text(i * img_width + img_width / 2,
                combined_grid.shape[1] + 15,  # Position below the images
                match_text,
                ha='center',
                va='top',
                fontsize=8,
                color=color,
                bbox=bbox_props)

    plt.suptitle("Verification Pairs (Top: Image 1, Bottom: Image 2)", y=1.02)
    plt.tight_layout()
    plt.subplots_adjust(bottom=0.05)
    plt.show()

if runtime["visualize"]:
    show_ver_dataset_samples(ver_val_loader)


# Model Architecture

## FAQ

### **What is the “very low early-deadline architecture” (mandatory early submission)?**

The *very low early-deadline architecture* is a simple **5-layer convolutional neural network (CNN)** designed to meet the early-submission requirements. Remember that the **maximum parameter budget for this homework is 30 million parameters**.

**Architecture details:**

* The network consists of **5 convolutional layers**.
* The **first convolutional layer**:

  * Output channels: **64**
  * Kernel size: **7**
  * Stride: **4**
* The **next four convolutional layers** have:

  * Output channels: **128 → 256 → 512 → 1024**
  * Kernel size: **3**
  * Stride: **2**
* Each convolutional layer is followed by:

  * **Batch Normalization**
  * **ReLU activation**

For creating convolutional layers, refer to the PyTorch documentation:
[https://pytorch.org/docs/stable/generated/torch.nn.Conv2d.html](https://pytorch.org/docs/stable/generated/torch.nn.Conv2d.html)

**Strided convolutions and padding:**

* Recall from lecture that a strided convolution can be viewed as a stride-1 convolution followed by downsampling.
* To preserve spatial alignment when using strided convolutions, choose:

  ```
  padding = kernel_size // 2
  ```
* Think about *why* this works: symmetric padding ensures the convolution remains centered over the input.

**Final pooling and flattening:**

* After the convolutional layers, apply **Adaptive Average Pooling** to reduce the spatial dimensions to **1 × 1**:

  * Use `AdaptiveAvgPool2d((1, 1))`
  * Documentation: [https://pytorch.org/docs/stable/generated/torch.nn.AdaptiveAvgPool2d.html](https://pytorch.org/docs/stable/generated/torch.nn.AdaptiveAvgPool2d.html)
* Finally, remove the trivial spatial dimensions (e.g., using `Flatten`).

For more layer options, see:
[https://pytorch.org/docs/stable/nn.html](https://pytorch.org/docs/stable/nn.html)

---

### **Why does this simple network use multiple downsampling convolutions?**

The input images are **112 × 112**. Each convolutional layer performs spatial downsampling.

* Downsampling by **2×** effectively **doubles the receptive field**, allowing neurons to capture information from a larger region of the image.
* Downsampling by **32×** in total is standard in many modern image models and provides a good balance between spatial resolution and semantic abstraction.

---

### **Why does this simple network use large channel sizes?**

Each time the spatial resolution is downsampled by **2×**, the computational cost drops by **4×** (assuming the number of channels stays constant).

To compensate:

* The number of channels is increased by **2×**, which increases computation by **4×**
* This keeps the overall computation roughly balanced across layers

Another intuition:

* As spatial resolution decreases, some spatial information is lost
* Increasing the number of channels helps preserve this information in the **feature (channel) dimension**

---


In [15]:
class ResidualBlock(torch.nn.Module):
    def __init__(self, c_in, c_out, stride):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(c_in, c_out, 3, stride=stride, padding=1),
            nn.BatchNorm2d(c_out),
            nn.PReLU(num_parameters=c_out),
            nn.Conv2d(c_out, c_out, 3, stride=1, padding=1),
            nn.BatchNorm2d(c_out),
        )

        if c_in == c_out and stride == 1:
            self.shortcut = nn.Identity()
        else:
            self.shortcut = nn.Sequential(
                nn.Conv2d(c_in, c_out, 1, stride=stride),
                nn.BatchNorm2d(c_out),
            )

        self.prelu = nn.PReLU()

    def forward(self, x):
        out = self.block(x) + self.shortcut(x)
        return self.prelu(out)

class Network(nn.Module):
    """
    Simple 5-layer CNN backbone for face classification and feature extraction.
    """

    def __init__(self, num_classes: int):
        super().__init__()
        self.backbone = nn.Sequential(
            # Stem
            nn.Conv2d(3, 64, 3, stride=1, padding=1),
            nn.BatchNorm2d(64),
            nn.PReLU(num_parameters=64),
            # Downsample
            nn.Conv2d(64, 64,3, stride=2, padding=1),
            nn.BatchNorm2d(64),
            nn.PReLU(num_parameters=64),
            # Stage 1
            ResidualBlock(64, 64, 1),
            ResidualBlock(64, 64, 1),
            # Stage 2
            ResidualBlock(64, 128, 2),
            ResidualBlock(128, 128, 1),
            ResidualBlock(128, 128, 1),
            # Stage 3
            ResidualBlock(128, 256, 2),
            ResidualBlock(256, 256, 1),
            ResidualBlock(256, 256, 1),
            # Stage 4
            ResidualBlock(256, 512, 2),
            ResidualBlock(512, 512, 1),
            # Adaptive Average Pooling
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(start_dim=1),
        )
        self.cls_layer = nn.Sequential(
            nn.Linear(512, num_classes)
        )

    def forward(self, x, return_feats: bool = False):
        """
        Args:
            x (Tensor): Input image batch of shape (B, 3, H, W)
            return_feats (bool): If True, return only feature embeddings

        Returns:
            dict with keys:
                - 'feats': feature embeddings
                - 'out': classification logits
        """
        feats = self.backbone(x)

        if return_feats:
            return feats

        out = self.cls_layer(feats)
        return {"feats": feats, "out": out}


# Initialize model
model = Network(num_classes=config["num_classes"]).to(DEVICE)

# Optional: model summary
summary(model, (3, 112, 112))

----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Conv2d-1         [-1, 64, 112, 112]           1,792
       BatchNorm2d-2         [-1, 64, 112, 112]             128
             PReLU-3         [-1, 64, 112, 112]              64
            Conv2d-4           [-1, 64, 56, 56]          36,928
       BatchNorm2d-5           [-1, 64, 56, 56]             128
             PReLU-6           [-1, 64, 56, 56]              64
            Conv2d-7           [-1, 64, 56, 56]          36,928
       BatchNorm2d-8           [-1, 64, 56, 56]             128
             PReLU-9           [-1, 64, 56, 56]              64
           Conv2d-10           [-1, 64, 56, 56]          36,928
      BatchNorm2d-11           [-1, 64, 56, 56]             128
         Identity-12           [-1, 64, 56, 56]               0
            PReLU-13           [-1, 64, 56, 56]               1
    ResidualBlock-14           [-1, 64,

# Loss | Optimizer | Scheduler

In [16]:
# ---------------------------------------------------
# Loss Function
# ---------------------------------------------------

# Cross-entropy loss is standard for multi-class classification.
# Label smoothing can help regularization when the number of classes is large.
# TODO: Initialize the criterion properly
criterion = nn.CrossEntropyLoss(label_smoothing=config['label_smoothing'])

# ---------------------------------------------------
# Optimizer
# ---------------------------------------------------

# SGD with momentum is a strong baseline for CNN training.
# Weight decay provides L2 regularization and helps prevent overfitting.

# TODO: Initialize the optimizer properly
optimizer = torch.optim.AdamW(model.parameters(), lr=config['lr'], weight_decay=config['weight_decay'])

# ---------------------------------------------------
# Learning Rate Scheduler
# ---------------------------------------------------

# Cosine annealing smoothly decays the learning rate to zero over training.
# TODO: Initialize the scheduler properly
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, config['epochs'])

# ---------------------------------------------------
# Mixed-Precision Training (FP16)
# ---------------------------------------------------

# GradScaler enables stable mixed-precision training on supported GPUs (e.g., T4, V100).
# This reduces memory usage and can significantly speed up training.
scaler = torch.amp.GradScaler(device='cuda', enabled=DEVICE == 'cuda')


# Metrics

In [17]:
class AverageMeter:
    """
    Tracks and computes the running average of a scalar metric.
    """

    def __init__(self):
        self.reset()

    def reset(self):
        self.val = 0.0
        self.avg = 0.0
        self.sum = 0.0
        self.count = 0

    def update(self, val, n: int = 1):
        self.val = float(val)
        self.sum += val * n
        self.count += n
        self.avg = self.sum / max(self.count, 1)


def topk_accuracy(logits: torch.Tensor, targets: torch.Tensor, topk=(1,)):
    """
    Computes top-k accuracy for the given logits and targets.

    Args:
        logits (Tensor): Model outputs of shape (B, C)
        targets (Tensor): Ground-truth labels of shape (B,)
        topk (tuple): Values of k for top-k accuracy

    Returns:
        List[Tensor]: Accuracy values in percentage for each k
    """
    maxk = min(max(topk), logits.size(1))
    batch_size = targets.size(0)

    _, preds = logits.topk(maxk, dim=1, largest=True, sorted=True)
    preds = preds.t()
    correct = preds.eq(targets.view(1, -1))

    accuracies = []
    for k in topk:
        k = min(k, maxk)
        correct_k = correct[:k].reshape(-1).float().sum(0)
        accuracies.append(correct_k * 100.0 / batch_size)

    return accuracies


def verification_metrics(labels, scores, fpr_targets=None):
    """
    Computes standard verification metrics: ACC, EER, AUC, and TPR@FPR.

    Args:
        labels (list or array): Binary labels (0 or 1)
        scores (list or array): Similarity scores
        fpr_targets (list, optional): Target FPR values for reporting TPR

    Returns:
        dict: Verification metrics
    """
    labels = np.asarray(labels)
    scores = np.asarray(scores)

    # TODO: Compute verificaion metrics
    # Hint: Read the documentation for sklearn.metrics.roc_curve
    # and use it to compute fpr and tpr
    fpr, tpr, _ = mt.roc_curve(labels, scores)

    # Hint: You can use interp1d from scipy.interpolate to create roc_interp
    roc_interp = interp1d(fpr, tpr, bounds_error=False, fill_value=(0.0, 1.0))

    # Equal Error Rate (EER)
    # Hint: You might find brentq from scipy.optimize helpful to compute EER
    eer = brentq(lambda x: 1.0 - x - roc_interp(x), 0.0, 1.0)

    # Area Under Curve (AUC)
    # Hint: Use mt.auc to compute AUC
    # Be careful to multiply by 100.0 to express as percentage
    auc = mt.auc(fpr, tpr) * 100.0

    # Accuracy (best threshold)
    # TODO: Compute TNR, pos_count, neg_count, and accuracy
    # Hint: Use fpr and tpr to compute TNR
    # Hint: pos_count is the number of positive labels (1s)
    # Hint: neg_count is the number of negative labels (0s)
    tnr = 1.0 - fpr
    pos_count = np.sum(labels==1)
    neg_count = np.sum(labels==0)
    acc = 100.0 * np.max((tpr * pos_count + tnr * neg_count) / (pos_count + neg_count))

    # TPR @ specific FPRs
    tpr_at_fpr = []
    if fpr_targets is not None:
        for fpr_val in fpr_targets:
            tpr_at_fpr.append(
                (f"TPR@FPR={fpr_val}", 100.0 * float(roc_interp(fpr_val)))
            )

    return {
        "ACC": acc,
        "EER": eer * 100,
        "AUC": auc,
        "TPRs": tpr_at_fpr,
    }


# Train and Validation Function

In [18]:
def train_epoch(
    model,
    dataloader,
    optimizer,
    scaler,
    device,
    criterion,
):
    """
    Runs one training epoch.

    Args:
        model (nn.Module): Model to train
        dataloader (DataLoader): Training dataloader
        optimizer (Optimizer): Optimizer
        scaler (GradScaler): AMP gradient scaler
        device (torch.device): Training device
        criterion (callable): Loss function

    Returns:
        tuple: (avg_accuracy, avg_loss)
    """
    global global_step
    model.train()

    loss_meter = AverageMeter()
    acc_meter = AverageMeter()

    progress = tqdm(
        dataloader,
        desc="Train",
        dynamic_ncols=True,
        leave=True,
    )

    for images, labels in progress:
        optimizer.zero_grad(set_to_none=True)

        # Move data to device
        images = images.to(device, non_blocking=True)

        if isinstance(labels, (tuple, list)):
            # For mixup / cutmix style labels
            targets1, targets2, lam = labels
            labels = (
                targets1.to(device, non_blocking=True),
                targets2.to(device, non_blocking=True),
                lam,
            )
        else:
            labels = labels.to(device, non_blocking=True)

        # Forward pass (mixed precision)
        with torch.amp.autocast(device_type=device, enabled=device == "cuda"):
            outputs = model(images)
            logits = outputs["out"]
            loss = criterion(logits, labels)

        # Backward + optimizer step (AMP-safe)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        global_step += 1

        # Metrics
        batch_loss = loss.item()
        loss_meter.update(batch_loss)

        with torch.no_grad():
            batch_acc = topk_accuracy(logits, labels, topk=(1,))[0].item()
            acc_meter.update(batch_acc)

        # Progress bar update
        progress.set_postfix(
            loss=f"{batch_loss:.4f} ({loss_meter.avg:.4f})",
            acc=f"{batch_acc:.2f}% ({acc_meter.avg:.2f}%)",
            lr=f"{optimizer.param_groups[0]['lr']:.6f}",
        )

    return acc_meter.avg, loss_meter.avg


In [19]:
@torch.no_grad()
def valid_epoch_cls(
    model,
    dataloader,
    device,
    criterion,
):
    """
    Runs one validation epoch for classification.

    Args:
        model (nn.Module): Trained model
        dataloader (DataLoader): Validation dataloader
        device (torch.device): Evaluation device
        criterion (callable): Loss function

    Returns:
        tuple: (avg_accuracy, avg_loss)
    """
    model.eval()

    loss_meter = AverageMeter()
    acc_meter = AverageMeter()

    progress = tqdm(
        dataloader,
        desc="Val (Cls)",
        dynamic_ncols=True,
        leave=True,
    )

    for images, labels in progress:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        # Forward pass (inference-only)
        outputs = model(images)
        logits = outputs["out"]
        loss = criterion(logits, labels)

        # Metrics
        batch_loss = loss.item()
        loss_meter.update(batch_loss)

        batch_acc = topk_accuracy(logits, labels, topk=(1,))[0].item()
        acc_meter.update(batch_acc)

        # Progress bar update
        progress.set_postfix(
            loss=f"{batch_loss:.4f} ({loss_meter.avg:.4f})",
            acc=f"{batch_acc:.2f}% ({acc_meter.avg:.2f}%)",
        )

    return acc_meter.avg, loss_meter.avg


In [20]:
gc.collect() # These commands help you when you face CUDA OOM error
torch.cuda.empty_cache()

# Verification Task

In [21]:
@torch.no_grad()
def valid_epoch_ver(model, pair_dataloader, device, fpr_targets=None):
    """
    Runs one validation epoch for verification (image pairs).

    Args:
        model (nn.Module): Trained model
        pair_dataloader (DataLoader): DataLoader yielding (img1, img2, label) tuples
        device (torch.device): Evaluation device
        fpr_targets (list, optional): List of FPRs for TPR reporting

    Returns:
        dict: Verification metrics including ACC, EER, AUC, TPR@FPR
    """
    model.eval()
    scores = []
    match_labels = []

    progress = tqdm(
        pair_dataloader,
        desc="Val (Veri)",
        dynamic_ncols=True,
        leave=True,
    )

    for images1, images2, labels in progress:
        # Move data to device
        images1 = images1.to(device, non_blocking=True)
        images2 = images2.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        # Forward pass
        images = torch.cat([images1, images2], dim=0)
        outputs = model(images)
        feats = F.normalize(outputs["feats"], dim=1)

        # Split features and compute similarity
        feats1, feats2 = feats.chunk(2, dim=0)
        similarity = F.cosine_similarity(feats1, feats2, dim=1)

        # Accumulate results
        scores.append(similarity.cpu().numpy())
        match_labels.append(labels.cpu().numpy())

    # Concatenate batch-wise results
    scores = np.concatenate(scores)
    match_labels = np.concatenate(match_labels)

    # Default FPR targets if not provided
    if fpr_targets is None:
        fpr_targets = [1e-4, 5e-4, 1e-3, 5e-3, 5e-2]

    # Compute verification metrics
    metric_dict = verification_metrics(match_labels, scores, fpr_targets)

    # Optional: print metrics
    print("Verification Metrics:", metric_dict)

    return metric_dict


# WandB

In [22]:
# Load only checkpoints you trust: full training state contains Python RNG objects.
resume_checkpoint = None
if runtime["resume_from"]:
    resume_checkpoint = torch.load(runtime["resume_from"], map_location="cpu", weights_only=False)
    saved_config = resume_checkpoint.get("config")
    if saved_config is not None:
        changed = [key for key in config if saved_config.get(key) != config[key]
                   and key not in {"display_name", "description"}]
        if changed:
            raise ValueError(f"Resume configuration mismatch: {changed}. Restore the original experiment settings.")
    saved_classes = resume_checkpoint.get("classes")
    if saved_classes is not None and saved_classes != cls_train_dataset.classes:
        raise ValueError("Checkpoint class mapping differs from the current dataset")
    # Validate architecture before connecting to an existing W&B run.
    model.load_state_dict(resume_checkpoint["model_state_dict"], strict=True)
    if "config" not in resume_checkpoint:
        print("Legacy checkpoint: configuration, scaler, RNG and historical best scores may be unavailable.")

if runtime["wandb_mode"] == "online":
    wandb_key = get_secret("WANDB_API_KEY", "WANDB_API_TOKEN")
    if wandb_key:
        wandb.login(key=wandb_key)
    else:
        wandb.login()  # Uses an existing login or prompts; choose offline/disabled to skip.


wandb: Currently logged in as: lukashann (lukashann-cmu) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [23]:
try:
    code_revision = subprocess.check_output(["git", "rev-parse", "HEAD"], stderr=subprocess.DEVNULL, text=True).strip()
except (OSError, subprocess.CalledProcessError):
    code_revision = None
import uuid
previous_run = (resume_checkpoint or {}).get("wandb", {})
saved_run_id = previous_run.get("id") or (resume_checkpoint or {}).get("run_id")
run = None
if runtime["wandb_mode"] != "disabled":
    if previous_run.get("id"):
        if previous_run.get("project") != runtime["wandb_project"]:
            raise ValueError("Resume must use the original W&B project")
        if runtime["wandb_entity"] and previous_run.get("entity") != runtime["wandb_entity"]:
            raise ValueError("Resume must use the original W&B entity")
    run = wandb.init(
        name=config["display_name"],
        project=runtime["wandb_project"],
        entity=previous_run.get("entity") or runtime["wandb_entity"],
        id=saved_run_id,
        resume="allow" if previous_run.get("id") and runtime["wandb_mode"] == "online" else None,
        mode=runtime["wandb_mode"],
        config={**config, "runtime": runtime, "code_revision": code_revision},
    )

# Use the actual W&B identity, or retain/generate an ID when logging is disabled.
runtime["run_id"] = run.id if run is not None else (saved_run_id or uuid.uuid4().hex)
if not runtime["run_id"] or Path(runtime["run_id"]).name != runtime["run_id"] or runtime["run_id"] in {".", ".."}:
    raise ValueError("Run ID must be a single directory name")
runtime["checkpoint_dir"] = str(Path(runtime["checkpoint_root"]) / runtime["run_id"])
print("Run ID:", runtime["run_id"])
print("Checkpoints:", runtime["checkpoint_dir"])


Run ID: fxz6v7n1
Checkpoints: /home/lukas/CMU/IDL/HW2/HW2P2/checkpoints/fxz6v7n1


# Checkpointing and Loading Model

In [24]:
import shutil
import warnings
checkpoint_dir = Path(runtime["checkpoint_dir"])
if not runtime["resume_from"] and (checkpoint_dir / "last.pth").exists():
    raise FileExistsError("This run already has a checkpoint. Set resume_from to continue it.")
checkpoint_dir.mkdir(parents=True, exist_ok=True)

def restore_rng(state):
    random.setstate(state["python"])
    np.random.set_state(state["numpy"])
    torch.set_rng_state(state["torch"])
    loader_generator.set_state(state["loader"])
    cuda_states = state.get("cuda", [])
    if cuda_states and torch.cuda.is_available():
        if len(cuda_states) == torch.cuda.device_count():
            torch.cuda.set_rng_state_all(cuda_states)
        else:
            warnings.warn("GPU count changed; CUDA RNG continuation will differ.")

def save_model(model, optimizer, scheduler, metrics, epoch, path):
    state = {
        "format_version": 2,
        "run_id": runtime["run_id"],
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict() if optimizer is not None else None,
        "scheduler_state_dict": scheduler.state_dict() if scheduler is not None else None,
        "scaler_state_dict": scaler.state_dict(),
        "metrics": metrics, "epoch": epoch, "global_step": global_step,
        "config": dict(config), "classes": cls_train_dataset.classes,
        "best_scores": dict(best_scores),
        "wandb": {"id": run.id, "project": run.project, "entity": run.entity} if run is not None else {},
        "code_revision": code_revision,
        "versions": {"torch": str(torch.__version__), "torchvision": str(torchvision.__version__)},
        "rng_state": {"python": random.getstate(), "numpy": np.random.get_state(),
                      "torch": torch.get_rng_state(), "loader": loader_generator.get_state(),
                      "cuda": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else []},
    }
    path = Path(path)
    temporary = path.with_suffix(path.suffix + ".tmp")
    torch.save(state, temporary)
    os.replace(temporary, path)
    if runtime["mirror_root"]:
        mirror = Path(runtime["mirror_root"]) / runtime["run_id"] / path.name
        if mirror.resolve() != path.resolve():
            mirror.parent.mkdir(parents=True, exist_ok=True)
            mirror_tmp = mirror.with_suffix(mirror.suffix + ".tmp")
            shutil.copy2(path, mirror_tmp)
            os.replace(mirror_tmp, mirror)
    if runtime["upload_checkpoints"] and run is not None and runtime["wandb_mode"] == "online":
        run.save(str(path), base_path=str(checkpoint_dir), policy="now")
    print(f"Checkpoint saved: {path}")

def load_model(model, optimizer=None, scheduler=None, path="./checkpoint.pth", device=None):
    """Load trusted weights/state for inference; runtime.resume_from handles full training resume."""
    checkpoint = torch.load(path, map_location="cpu", weights_only=False)
    if device is not None:
        model.to(device)
    model.load_state_dict(checkpoint["model_state_dict"])
    if optimizer is not None and checkpoint.get("optimizer_state_dict") is not None:
        optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
    if scheduler is not None and checkpoint.get("scheduler_state_dict") is not None:
        scheduler.load_state_dict(checkpoint["scheduler_state_dict"])
    return model, optimizer, scheduler, checkpoint.get("epoch", 0), checkpoint.get("metrics", {})


# Experiments

In [25]:
# Resume only at completed epoch boundaries. Total epochs is the original training horizon.
start_epoch = 0
global_step = 0
best_scores = {"cls": 0.0, "ret": 0.0, "combined": 0.0}
if resume_checkpoint is not None:
    for key in ("optimizer_state_dict", "scheduler_state_dict"):
        if resume_checkpoint.get(key) is None:
            raise ValueError(f"Training resume requires {key}; use load_model for weights-only inference.")
    optimizer.load_state_dict(resume_checkpoint["optimizer_state_dict"])
    scheduler.load_state_dict(resume_checkpoint["scheduler_state_dict"])
    if resume_checkpoint.get("scaler_state_dict"):
        scaler.load_state_dict(resume_checkpoint["scaler_state_dict"])
    start_epoch = resume_checkpoint["epoch"] + 1
    global_step = resume_checkpoint.get("global_step", start_epoch * len(cls_train_loader))
    legacy_metrics = resume_checkpoint.get("metrics", {})
    best_scores.update(resume_checkpoint.get("best_scores", {
        "cls": legacy_metrics.get("valid_cls_acc", 0.0),
        "ret": legacy_metrics.get("valid_ret_acc", 0.0),
        "combined": legacy_metrics.get("valid_combined_score", 0.0),
    }))
    if resume_checkpoint.get("rng_state"):
        restore_rng(resume_checkpoint["rng_state"])
    resume_checkpoint = None  # Release CPU copies of model and optimizer tensors.
    gc.collect()
print(f"Starting at epoch {start_epoch + 1}; target total epochs: {config['epochs']}")

for epoch in range(start_epoch, config["epochs"]):
    print(f"\n=== Epoch {epoch + 1}/{config['epochs']} ===")

    curr_lr = optimizer.param_groups[0]["lr"]
    # -----------------------------
    # Train
    # -----------------------------
    train_cls_acc, train_loss = train_epoch(
        model, cls_train_loader, optimizer, scaler, DEVICE, criterion
    )
    print(
        f"Train | Cls Acc: {train_cls_acc:.4f}% | Loss: {train_loss:.4f} | LR: {curr_lr:.6f}"
    )

    metrics = {
        "train_cls_acc": train_cls_acc,
        "train_loss": train_loss,
        "lr": curr_lr,
    }

    # -----------------------------
    # Classification Validation
    # -----------------------------
    valid_cls_acc, valid_loss = valid_epoch_cls(
        model, cls_val_loader, DEVICE, criterion
    )

    print(f"Val (Cls) | Acc: {valid_cls_acc:.4f}% | Loss: {valid_loss:.4f}")

    metrics.update({
        "valid_cls_acc": valid_cls_acc,
        "valid_loss": valid_loss,
    })

    # -----------------------------
    # Verification / Retrieval Validation
    # -----------------------------
    valid_ret_metrics = valid_epoch_ver(model, ver_val_loader, DEVICE)
    valid_ret_acc = valid_ret_metrics["ACC"]
    valid_eer = valid_ret_metrics["EER"]
    print(f"Val (Veri) | ACC: {valid_ret_acc:.4f}% | EER: {valid_eer:.4f}%")
    metrics.update({
        "valid_ret_acc": valid_ret_acc,
        "valid_eer": valid_eer,
    })

    # -----------------------------
    # Combined Score Calculation
    # -----------------------------
    valid_combined_score = 0.5 * valid_cls_acc + 0.5 * (100.0 - valid_eer)
    print(f"Val (Combined) | Score: {valid_combined_score:.4f}%")
    metrics.update({
        "valid_combined_score": valid_combined_score
    })

    # -----------------------------
    # Log metrics
    # -----------------------------
    if "run" in globals() and run is not None:
        run.log({**metrics, "epoch": epoch + 1, "global_step": global_step})

    # -----------------------------
    # Step scheduler
    # Prepare LR for NEXT epoch
    # -----------------------------
    if scheduler is not None:
        scheduler.step()

    # Update all best scores before saving any checkpoint, including last.pth.
    current_scores = {"cls": valid_cls_acc, "ret": valid_ret_acc, "combined": valid_combined_score}
    improved = {key: value >= best_scores[key] for key, value in current_scores.items()}
    for key, value in current_scores.items():
        best_scores[key] = max(best_scores[key], value)
    save_model(model, optimizer, scheduler, metrics, epoch, checkpoint_dir / "last.pth")
    for key in current_scores:
        if improved[key]:
            save_model(model, optimizer, scheduler, metrics, epoch, checkpoint_dir / f"best_{key}.pth")

if run is not None:
    run.finish()  # Flush pending metrics/checkpoint uploads before ending the session.


Starting at epoch 1; target total epochs: 20

=== Epoch 1/20 ===


Train: 100%|██████████| 1349/1349 [02:54<00:00,  7.71it/s, acc=0.00% (0.24%), loss=6.8195 (7.2683), lr=0.001000]


Train | Cls Acc: 0.2409% | Loss: 7.2683 | LR: 0.001000


Val (Cls): 100%|██████████| 135/135 [00:07<00:00, 17.24it/s, acc=0.00% (0.94%), loss=6.4643 (6.7070)]


Val (Cls) | Acc: 0.9375% | Loss: 6.7070


...

Val (Cls): 100%|██████████| 135/135 [00:08<00:00, 16.42it/s, acc=97.92% (87.04%), loss=1.6178 (1.9266)]


Val (Cls) | Acc: 87.0448% | Loss: 1.9266


Val (Veri): 100%|██████████| 16/16 [00:01<00:00, 10.24it/s]


Verification Metrics: {'ACC': np.float64(94.39999999999999), 'EER': 5.697445972577729, 'AUC': 97.83129734033827, 'TPRs': [('TPR@FPR=0.0001', 83.5030549898167), ('TPR@FPR=0.0005', 83.5030549898167), ('TPR@FPR=0.001', 83.5030549898167), ('TPR@FPR=0.005', 85.94704684317719), ('TPR@FPR=0.05', 93.48268839103869)]}
Val (Veri) | ACC: 94.4000% | EER: 5.6974%
Val (Combined) | Score: 90.6737%
Checkpoint saved: /home/lukas/CMU/IDL/HW2/HW2P2/checkpoints/fxz6v7n1/last.pth
Checkpoint saved: /home/lukas/CMU/IDL/HW2/HW2P2/checkpoints/fxz6v7n1/best_cls.pth
Checkpoint saved: /home/lukas/CMU/IDL/HW2/HW2P2/checkpoints/fxz6v7n1/best_combined.pth


epoch,▁▁▂▂▂▃▃▄▄▄▅▅▅▆▆▇▇▇██
global_step,▁▁▂▂▂▃▃▄▄▄▅▅▅▆▆▇▇▇██
lr,████▇▇▇▆▆▅▄▄▃▃▂▂▂▁▁▁
train_cls_acc,▁▁▃▅▆▆▇▇▇███████████
train_loss,█▆▅▃▃▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁
valid_cls_acc,▁▂▄▅▆▆▇▇▇▇██████████
valid_combined_score,▁▃▄▅▆▇▇▇▇▇██████████
valid_eer,█▄▃▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁
valid_loss,█▆▄▃▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁
valid_ret_acc,▁▅▆▇▇▇██▇███████████
epoch,20


# Inference & Submission

## Kaggle Submission Instructions

* Run the code block below to **automatically generate** the submission file for the competition.

In [26]:
#### DO NOT MODIFY ####
# !git clone https://github.com/CMU-IDeeL/HW2P2-Student-Utils.git
# !mv HW2P2-Student-Utils/utils .
# !rm -rf HW2P2-Student-Utils
# from utils import (
#     run_cls_inference_file_order,
#     run_ver_inference,
#     write_submission_bundle,
# )

# cls_predictions = run_cls_inference_file_order(
#     model=model,
#     cls_test_root=cls_data_dir + "/test",
#     transform=val_transforms,
#     device=DEVICE,
#     batch_size=config["batch_size"],
#     logits_key="out",
# )

# ver_scores = run_ver_inference(
#     model=model,
#     pair_dataloader=ver_test_loader,
#     device=DEVICE,
#     embedding_key="feats",
# )

# submission_rows = write_submission_bundle(
#     cls_predictions=cls_predictions,
#     ver_scores=ver_scores,
#     submission_path="submission.csv",
#     cls_component_path="cls_submission.csv",
#     ver_component_path="ver_submission.csv",
# )

# print("Saved submission.csv")
# print(f"  Classification predictions: {len(cls_predictions)}")
# print(f"  Verification scores:        {len(ver_scores)}")
# print(f"  Total rows:                  {len(submission_rows)}")
# print(pd.DataFrame(submission_rows).head())

* Preview the merged `submission.csv` file created in the cell above

In [27]:
# submission_preview = pd.read_csv("submission.csv")
# print(submission_preview.head())
# print(f"Submission rows: {len(submission_preview)}")
# print(f"Columns: {list(submission_preview.columns)}")


* Run the code block below to **automatically submit** the generated submission file to the competition.

In [28]:
# api = get_kaggle_api()
# api.competition_submit(file_name="submission.csv", message="<ENTER_YOUR_MESSAGE_HERE>", competition="hw-2-p-2-fall-2026-student-competition")



#### **Manual submission (optional)**

If automatic submission is unavailable, download the CSV and submit it manually using one of the options below.

##### **Kaggle**

* After running the cell above, the file **`submission.csv`** will be saved in:
  **Right sidebar → Output → `kaggle/working/`**
* If the file does not appear right away, click the **refresh icon** in the top-right corner of the Output panel.
* Once the file is visible, **right-click → Download**.
* Upload the downloaded file to the **Kaggle competition submission page**.

---

##### **Colab**

```python
from google.colab import files
files.download("submission.csv")
```

##### **PSC**

* Open the left file browser.
* Navigate to:
  `/jet/home/<your_username>/`
* Locate **`submission.csv`**.
* Right-click the file and select **Download**.
* If the file does not appear immediately, refresh the file browser.

## 📦 Gradescope Submission Instructions (Read Carefully)

This section guides you through creating and submitting your **final Gradescope code submission**.
Please follow each step in order. Skipping steps may result in an incomplete or invalid submission.

### ⏰ Important Deadlines

* **Kaggle Final Deadline:** See the course schedule
* **Gradescope Code Submission Deadline:**
  **48 hours after** the Kaggle deadline (or the same day as your Slack Kaggle submission, if applicable)

You must submit **both**:

1. A valid Kaggle submission
2. A complete Gradescope code submission zip

### Step 1: Generate Your Model Metadata File

Please assign your **final trained model** to the global variable `MODEL`.

We will use this variable to automatically generate a file named
`model_metadata.json`, which records important information about your model (such as parameter count and architecture).

#### 🔴 Important Requirements

* **`MODEL` must be the exact model used for your best Kaggle submission**
* This step is **required** for grading and audit purposes
* If the model does not match your Kaggle submission, your score may be invalidated

#### 🕒 When to Do This

* Run this cell **only after you have finished training** your final model
* Do **not** retrain or modify the model after assigning it to `MODEL`



In [29]:
MODEL = model # TODO: Initialize to your tained model

### Step 2: Complete the README Section


Fill in the `README` variable with a brief description of your work:

* **Model**: Architecture and key design choices
* **Training Strategy**: Optimizer, scheduler, loss, etc.
* **Augmentations**: If used (omit if not)
* **Notebook Execution**: Any instructions needed to run your notebook

This README will be included in your submission zip.

In [30]:
README = """
- **Model**: Model archtiecture description. Anything unique? Any specific architecture shapes or strategies?
- **Training Strategy**: optimizer + scheduler + loss function + any other unique ideas
- **Augmentations**: augmentations if used. If augmentations weren't used, then ignore
- **Notebook Execution**: Any instructions required to run your notebook.
"""

### Step 3: Provide Required Credentials

Ensure the following variables are correctly set:

* `KAGGLE_USERNAME`
* `KAGGLE_API_KEY`
* `WANDB_API_KEY`
* `WANDB_USERNAME_OR_TEAMNAME`
* `WANDB_PROJECT`

These are used **only** to:

* Fetch your official Kaggle score
* Export your top WandB runs for grading

⚠️ Do **not** share your API keys publicly.


In [31]:
# @title
KAGGLE_USERNAME = "<your-username>" # TODO
KAGGLE_API_KEY  = "<your-key>"      # TODO
WANDB_API_KEY   = "<ENTER-YOUR-WANDB-API-KEY-HERE>" # TODO
WANDB_USERNAME_OR_TEAMNAME = "<ENTER-YOUR-WANDB-USERNAME-OR-TEAMNAME-HERE>" # TODO: Put your username-or-team-name here
WANDB_PROJECT              = "<ENTER-YOUR-WANDB-PROJECT-NAME-HERE>" # TODO: Put your project-name

### Step 4: Set File Paths Correctly

You must provide absolute path to your **final notebook**

### Platform-specific guidance

**Colab**

* Right-click a file in the left file pane → **Copy path**
* Paths typically start with `/content/...`

**Kaggle**

* Download your notebook (`File → Download Notebook`)
* Upload it via **Upload Input → Upload Model**
* Copy paths from the right sidebar (`/kaggle/working/...`)

**PSC / Linux**

* Locate files under `/jet/home/<your_username>/`
* Use `!ls` to confirm paths

Paste the correct path below.

In [32]:
NOTEBOOK_PATH = "<ENTER-YOUR-NOTEBOOK-PATH-HERE>" # TODO

### Step 5: (Optional) Additional Files

If you have extra files you want to include (e.g., config files, scripts), add their paths to `ADDITIONAL_FILES`, Otherwise, leave this list empty.

In [33]:
ADDITIONAL_FILES = []

### Step 6: Generate the Final Submission Zip

Before running the submission cell:

* Ensure `ACKNOWLEDGED = True`
* Double-check all paths and credentials
* Confirm your Kaggle submission exists

Then run the cell below.
This should generate a `HW2P2_final_submission.zip` which includes:
* Your notebook
* `model_metadata_*.json`
* README
* WandB run exports
* Kaggle metadata
* Academic integrity acknowledgement

In [34]:
# #### DO NOT MODIFY ####
# !git clone https://github.com/CMU-IDeeL/HWP2-Submission-Backend-Student.git
# !mv HWP2-Submission-Backend-Student/submission .
# !rm -rf HWP2-Submission-Backend-Student
# from submission.submission_config import SubmissionConfig
# from submission.backend_config import BackendConfig, HW2P2_BACKEND_CONFIG
# from submission.main import create_submission_zip

# create_submission_zip(
#     cfg = SubmissionConfig(
#         model = MODEL,
#         kaggle_username  = KAGGLE_USERNAME,
#         kaggle_api_key   = KAGGLE_API_KEY,
#         wandb_api_key    = WANDB_API_KEY,
#         wandb_entity     = WANDB_USERNAME_OR_TEAMNAME,
#         wandb_project    = WANDB_PROJECT,
#         acknowledged     = ACKNOWLEDGED,
#         readme           = README,
#         notebook_path    = NOTEBOOK_PATH,
#         additional_files = ADDITIONAL_FILES
#     ),
#     backend_cfg = HW2P2_BACKEND_CONFIG
# )

### Step 7: Upload to Gradescope

1. Download `HW2P2_final_submission.zip`
2. Upload it to **Gradescope**
3. Verify the upload succeeds

⚠️ Submitting an incomplete zip or modifying backend code may result in grading penalties or an Academic Integrity Violation.

---

### ✅ Final Checklist

Before submitting, confirm:

* [ ] Kaggle username is valid
* [ ] `model_metadata_*.json` matches your best run
* [ ] README is complete
* [ ] File paths are correct
* [ ] `ACKNOWLEDGED = True`
* [ ] Final zip downloads successfully

---